In [ ]:
import time
import tracemalloc

import numpy as np
from scipy.optimize import curve_fit

import matplotlib.pyplot as plt

plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['savefig.edgecolor'] = 'white'

plt.rcParams["figure.dpi"] = 500
plt.rcParams["savefig.dpi"] = 500

from pathlib import Path
import importlib.util
import sys
import tempfile
import urllib.request

_reference_util_path = next(
    (
        candidate
        for base in (Path.cwd(), *Path.cwd().parents)
        for candidate in (
            base / "capitulo4" / "runtime" / "util.py",
            base / "runtime" / "util.py",
        )
        if candidate.exists()
    ),
    None,
)
if _reference_util_path is None:
    _reference_util_path = Path(tempfile.gettempdir()) / "capitulo4_reference_util.py"
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Notas-a-Mano-serie-de-libros/"
        "3_notas-a-mano-sobre-analisis-de-complejidad-computacional/"
        "main/capitulo4/runtime/util.py",
        _reference_util_path,
    )
_reference_spec = importlib.util.spec_from_file_location(
    "capitulo4_reference_util", _reference_util_path
)
_reference_util = importlib.util.module_from_spec(_reference_spec)
sys.modules[_reference_spec.name] = _reference_util
_reference_spec.loader.exec_module(_reference_util)
graficar_complejidad = _reference_util.graficar_complejidad
modelo_constante = _reference_util.modelo_constante
modelo_lineal = _reference_util.modelo_lineal
modelo_cuadratico = _reference_util.modelo_cuadratico


In [ ]:
def fib_big(n: int) -> int:
    if n <= 1:
        return n
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

def medir_tiempo_fib(func, sizes, n_iter):
    tiempos = np.zeros(len(sizes), dtype=float)
    for i, n in enumerate(sizes):
        acumulado = 0.0
        for _ in range(n_iter):
            inicio = time.perf_counter()
            func(int(n))
            acumulado += (time.perf_counter() - inicio)
        tiempos[i] = acumulado / n_iter
    return tiempos

def medir_memoria_fib(func, sizes, n_iter):
    picos = np.zeros(len(sizes), dtype=float)
    for i, n in enumerate(sizes):
        acumulado = 0.0
        for _ in range(n_iter):
            tracemalloc.start()
            func(int(n))
            _, peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()
            acumulado += peak
        picos[i] = acumulado / n_iter
    return picos



In [ ]:
n_ejecuciones = 10
sizes = np.arange(1, 20000, 500)
x = sizes

tiempos = medir_tiempo_fib(fib_big, sizes, n_ejecuciones)

In [ ]:
params_tiempo_lineal = curve_fit(modelo_lineal, sizes, tiempos)[0]
tiempos_lineal_ajustados = modelo_lineal(sizes, *params_tiempo_lineal)

graficar_complejidad(
    x=sizes,
    y_experimental=tiempos,
    y_teorico=tiempos_lineal_ajustados,
    nombre_archivo="fib_big_tiempo_lineal.png",
    ylabel="Tiempo de ejecución [s]",
    funcion="T(n)"
)

In [ ]:

params_tiempo = curve_fit(modelo_cuadratico, sizes**2, tiempos)[0]
tiempos_ajustados = modelo_cuadratico(sizes**2, *params_tiempo)

graficar_complejidad(
    x=sizes,
    y_experimental=tiempos,
    y_teorico=tiempos_ajustados,
    nombre_archivo="fib_big_tiempo.png",
    ylabel="Tiempo de ejecución [s]",
    funcion="T(n)"
)

In [ ]:
n_ejecuciones = 10
sizes = np.arange(1, 10000, 500)
recursos = medir_memoria_fib(fib_big, sizes, n_ejecuciones)

params_memoria = curve_fit(modelo_cuadratico, sizes, recursos)[0]
recursos_ajustados = modelo_cuadratico(sizes, *params_memoria)

graficar_complejidad(
    x=sizes,
    y_experimental=recursos,
    y_teorico=recursos_ajustados,
    nombre_archivo="fib_big_espacio.png",
    ylabel="Consumo de memoria [bytes]",
    funcion="S(n)"
)